
# GeoAI Assignment Notebook: Supervised Machine Learning with Vienna Airbnb Data

This notebook is the **graded assignment** for the introductory ML part of the GeoAI course.

## What you are assessed on
There are **10 questions × 1 point each = 10 points**.

- **8 technical questions** ask you to write code.
- **1 point** is for a successful GitHub Classroom submission.
- **1 point** is for short assignment feedback.

## Scope
Stay within the methods already covered in class:
- **Classification:** Logistic Regression, KNN, Decision Tree were shown in the tutorial.  
  In this assignment you will implement **SVM** and **Random Forest**.
- **Regression:** Linear Regression, Lasso, and Decision Tree were shown in the tutorial.  
  In this assignment you will implement **Ridge** and **Random Forest**.
- We do **not** use PCA here.

## Important rules
1. **Do not rename the required variables.** The grader will look for the variable names written in the notebook.
2. Keep the provided setup cells unchanged unless the instructions explicitly ask you to edit them.
3. Run the notebook from top to bottom before your final commit.
4. Commit and push your final version to **GitHub Classroom**.



## Recommended repo structure for GitHub Classroom

Place the notebook and data in a simple structure like this:

```text
your-repo/
├─ GeoAI_Assignment_Student.ipynb
└─ data/
   └─ vienna_airbnb_assignment_ready.csv
```

The notebook below tries these locations automatically:
- `data/vienna_airbnb_assignment_ready.csv`
- `vienna_airbnb_assignment_ready.csv`
- `/mnt/data/vienna_airbnb_assignment_ready.csv`


In [ ]:

# ============================================================
# 0. Imports and global settings
# ============================================================
from pathlib import Path
import re
import math
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression, LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

warnings.filterwarnings("ignore")
RANDOM_STATE = 42


In [ ]:

# ============================================================
# 1. Load the prepared assignment dataset
# ============================================================
possible_paths = [
    Path("data/vienna_airbnb_assignment_ready.csv"),
    Path("vienna_airbnb_assignment_ready.csv"),
]

data_path = None
for p in possible_paths:
    if p.exists():
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError(
        "Could not find vienna_airbnb_assignment_ready.csv. "
        "Place it in data/ or in the same folder as this notebook."
    )

df = pd.read_csv(data_path)
print("Loaded:", data_path.resolve())
print("Shape:", df.shape)
df.head()



## Provided feature lists

These are the same core features used in the tutorials.
- For **classification**, do **not** use `review_scores_rating` as an input because the target `highly_rated` was created from it.
- For **regression**, you may use `review_scores_rating` as an input when predicting price.


In [ ]:

# ============================================================
# 2. Shared feature lists
# ============================================================
numeric_features_cls = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms_num",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_365",
    "number_of_reviews",
    "reviews_per_month",
    "host_identity_verified_num",
    "amenity_count",
    "calculated_host_listings_count",
]

categorical_features = [
    "room_type",
    "property_type",
    "neighbourhood_joined",
    "host_response_time",
]

numeric_features_reg = numeric_features_cls + ["review_scores_rating"]

print("Classification numeric features:", len(numeric_features_cls))
print("Regression numeric features:", len(numeric_features_reg))
print("Categorical features:", len(categorical_features))



# Part A. Classification

In the tutorial you already saw:
- Logistic Regression
- KNN
- Decision Tree

Here you will implement:
- **SVM**
- **Random Forest Classifier**



## Q1 [1 point]
Create the classification feature matrix and target, then split the data into training and test sets.

**Tasks**
- Use the provided feature lists.
- Create `X_cls` and `y_cls`.
- Use `train_test_split(...)` with:
  - `test_size=0.25`
  - `random_state=RANDOM_STATE`
  - `stratify=y_cls`
- Save the outputs as:
  - `Xc_train`, `Xc_test`, `yc_train`, `yc_test`


In [ ]:

# Q1 - YOUR CODE HERE

class_df = df.dropna(subset=["highly_rated"]).copy()
feature_cols_cls = numeric_features_cls + categorical_features

# TODO: create X_cls and y_cls
X_cls = None
y_cls = None

# TODO: split into train and test data
Xc_train, Xc_test, yc_train, yc_test = None, None, None, None


In [ ]:

# Q1 - quick check
assert isinstance(X_cls, pd.DataFrame), "Q1: X_cls should be a pandas DataFrame."
assert isinstance(y_cls, pd.Series), "Q1: y_cls should be a pandas Series."
assert set(pd.Series(y_cls).dropna().unique()).issubset({0, 1}), "Q1: y_cls should be binary (0/1)."
assert len(Xc_train) + len(Xc_test) == len(X_cls), "Q1: train and test rows should add up."
assert len(yc_train) + len(yc_test) == len(y_cls), "Q1: train and test targets should add up."
print("Q1 check passed.")



## Q2 [1 point]
Build the preprocessing pipeline for classification.

**Tasks**
- Create `numeric_transformer_cls`
- Create `categorical_transformer_cls`
- Combine them into `preprocessor_cls`

Use:
- median imputation + standard scaling for numeric columns
- most-frequent imputation + one-hot encoding for categorical columns


In [ ]:

# Q2 - YOUR CODE HERE

# TODO: build the numeric transformer
numeric_transformer_cls = None

# TODO: build the categorical transformer
categorical_transformer_cls = None

# TODO: combine them in a ColumnTransformer called preprocessor_cls
preprocessor_cls = None


In [ ]:

# Q2 - quick check
assert isinstance(preprocessor_cls, ColumnTransformer), "Q2: preprocessor_cls should be a ColumnTransformer."
print("Q2 check passed.")



### Tutorial baseline results for comparison

These are approximate results from the tutorial models.  
Your exact values may differ slightly depending on software versions.

- Logistic Regression: accuracy ≈ 0.651, F1 ≈ 0.664
- KNN: accuracy ≈ 0.634, F1 ≈ 0.636
- Decision Tree: accuracy ≈ 0.667, F1 ≈ 0.652


In [ ]:

classification_baselines = pd.DataFrame([{'model': 'Logistic Regression', 'accuracy': 0.651, 'f1': 0.664}, {'model': 'KNN', 'accuracy': 0.634, 'f1': 0.636}, {'model': 'Decision Tree', 'accuracy': 0.667, 'f1': 0.652}])
classification_baselines


In [ ]:

# ============================================================
# Helper for classification evaluation
# ============================================================
def evaluate_classifier(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
    }



## Q3 [1 point]
Train and evaluate an **SVM classifier** inside a pipeline.

**Tasks**
- Create a pipeline called `svm_clf`
- Use `preprocessor_cls` followed by `SVC(...)`
- Use:
  - `kernel="rbf"`
  - `C=1.0`
  - `random_state=RANDOM_STATE`
- Evaluate it with `evaluate_classifier(...)`
- Save the result dictionary in `svm_result`


In [ ]:

# Q3 - YOUR CODE HERE

# TODO: create the SVM pipeline
svm_clf = None

# TODO: evaluate the model and store the metrics in svm_result
svm_metrics = None

svm_result = {
    "model": "SVM",
    "accuracy": None,
    "f1": None,
}


In [ ]:

# Q3 - quick check
assert isinstance(svm_result, dict), "Q3: svm_result should be a dictionary."
assert svm_result["model"] == "SVM", "Q3: set model name to 'SVM'."
assert 0 <= svm_result["accuracy"] <= 1, "Q3: accuracy should be between 0 and 1."
assert 0 <= svm_result["f1"] <= 1, "Q3: f1 should be between 0 and 1."
print("Q3 check passed.")



## Q4 [1 point]
Train and evaluate a **Random Forest classifier**, then try one simple hyperparameter change.

**Tasks**
1. Create a pipeline called `rf_clf`
2. Use:
   - `n_estimators=200`
   - `max_depth=10`
   - `random_state=RANDOM_STATE`
3. Evaluate it and store the result in `rf_clf_result`
4. Create a small table called `rf_depth_results_cls` comparing **two** `max_depth` values:
   - 5
   - 10
5. Write **one short sentence** in `rf_cls_comment` describing what changed


In [ ]:

# Q4 - YOUR CODE HERE

# TODO: create and evaluate the main random forest classifier
rf_clf = None
rf_clf_metrics = None

rf_clf_result = {
    "model": "Random Forest Classifier",
    "accuracy": None,
    "f1": None,
}

# TODO: compare two max_depth values (5 and 10)
rf_depth_results_cls = None

# TODO: write one short interpretation sentence
rf_cls_comment = ""


In [ ]:

# Q4 - quick check
assert isinstance(rf_clf_result, dict), "Q4: rf_clf_result should be a dictionary."
assert 0 <= rf_clf_result["accuracy"] <= 1, "Q4: accuracy should be between 0 and 1."
assert 0 <= rf_clf_result["f1"] <= 1, "Q4: f1 should be between 0 and 1."
assert isinstance(rf_depth_results_cls, pd.DataFrame), "Q4: rf_depth_results_cls should be a DataFrame."
assert len(rf_depth_results_cls) == 2, "Q4: compare exactly two max_depth values."
assert len(rf_cls_comment.strip()) >= 15, "Q4: write a short interpretation sentence."
print("Q4 check passed.")


In [ ]:

classification_results_final = pd.concat(
    [
        classification_baselines,
        pd.DataFrame([svm_result, rf_clf_result]),
    ],
    ignore_index=True,
).sort_values(by="f1", ascending=False)

classification_results_final



# Part B. Regression

In the tutorial you already saw:
- Linear Regression
- Lasso Regression
- Decision Tree Regressor

Here you will implement:
- **Ridge Regression**
- **Random Forest Regressor**


In [ ]:

# ============================================================
# Helper for price parsing
# ============================================================
def parse_price(value):
    if pd.isna(value):
        return np.nan

    s = str(value).strip()
    s = re.sub(r"[^0-9.\-]", "", s)

    if s == "":
        return np.nan

    try:
        return float(s)
    except ValueError:
        return np.nan



## Q5 [1 point]
Prepare the regression target and split the data.

**Tasks**
- Create `price_num` using `parse_price(...)`
- Remove rows where `price_num` is missing or not positive
- Remove extreme price outliers using the **upper IQR fence**
- Create `X_reg` and `y_reg`
- Split into train and test sets using:
  - `test_size=0.25`
  - `random_state=RANDOM_STATE`
- Save the outputs as:
  - `Xr_train`, `Xr_test`, `yr_train`, `yr_test`


In [ ]:

# Q5 - YOUR CODE HERE

reg_df = df.copy()

# TODO: create a numeric target called price_num
reg_df["price_num"] = None

# TODO: remove missing and non-positive prices

# TODO: compute the IQR upper fence and remove high outliers
upper_fence = None

feature_cols_reg = numeric_features_reg + categorical_features

# TODO: create X_reg and y_reg
X_reg = None
y_reg = None

# TODO: split into train and test data
Xr_train, Xr_test, yr_train, yr_test = None, None, None, None


In [ ]:

# Q5 - quick check
assert "price_num" in reg_df.columns, "Q5: reg_df must contain price_num."
assert np.issubdtype(reg_df["price_num"].dtype, np.number), "Q5: price_num should be numeric."
assert isinstance(X_reg, pd.DataFrame), "Q5: X_reg should be a DataFrame."
assert isinstance(y_reg, pd.Series), "Q5: y_reg should be a Series."
assert len(Xr_train) + len(Xr_test) == len(X_reg), "Q5: train and test rows should add up."
print("Q5 check passed.")



## Q6 [1 point]
Build the preprocessing pipeline for regression.

**Tasks**
- Create `numeric_transformer_reg`
- Create `categorical_transformer_reg`
- Combine them into `preprocessor_reg`


In [ ]:

# Q6 - YOUR CODE HERE

# TODO: build the numeric transformer
numeric_transformer_reg = None

# TODO: build the categorical transformer
categorical_transformer_reg = None

# TODO: combine them in a ColumnTransformer called preprocessor_reg
preprocessor_reg = None


In [ ]:

# Q6 - quick check
assert isinstance(preprocessor_reg, ColumnTransformer), "Q6: preprocessor_reg should be a ColumnTransformer."
print("Q6 check passed.")



### Tutorial baseline results for comparison

Approximate baseline results from the tutorial:
- Linear Regression: MAE ≈ 25.84, RMSE ≈ 33.69, R² ≈ 0.400
- Lasso Regression: MAE ≈ 25.98, RMSE ≈ 34.12, R² ≈ 0.384
- Decision Tree Regressor: MAE ≈ 26.98, RMSE ≈ 36.64, R² ≈ 0.290


In [ ]:

regression_baselines = pd.DataFrame([{'model': 'Linear Regression', 'mae': 25.84, 'rmse': 33.69, 'r2': 0.4}, {'model': 'Lasso Regression', 'mae': 25.98, 'rmse': 34.12, 'r2': 0.384}, {'model': 'Decision Tree Regressor', 'mae': 26.98, 'rmse': 36.64, 'r2': 0.29}])
regression_baselines


In [ ]:

# ============================================================
# Helper for regression evaluation
# ============================================================
def evaluate_regressor(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    return {
        "mae": mean_absolute_error(y_test, predictions),
        "rmse": math.sqrt(mean_squared_error(y_test, predictions)),
        "r2": r2_score(y_test, predictions),
    }



## Q7 [1 point]
Train and evaluate a **Ridge regression** pipeline, then try one hyperparameter change.

**Tasks**
1. Create a pipeline called `ridge_reg`
2. Use `Ridge(alpha=1.0, random_state=RANDOM_STATE)`
3. Evaluate it and store the result in `ridge_result`
4. Create a small table `ridge_alpha_results` comparing **two** alpha values:
   - 0.1
   - 1.0
5. Write **one short sentence** in `ridge_comment`


In [ ]:

# Q7 - YOUR CODE HERE

# TODO: create and evaluate the main ridge model
ridge_reg = None
ridge_metrics = None

ridge_result = {
    "model": "Ridge Regression",
    "mae": None,
    "rmse": None,
    "r2": None,
}

# TODO: compare two alpha values (0.1 and 1.0)
ridge_alpha_results = None

# TODO: write one short interpretation sentence
ridge_comment = ""


In [ ]:

# Q7 - quick check
assert isinstance(ridge_result, dict), "Q7: ridge_result should be a dictionary."
assert ridge_result["model"] == "Ridge Regression", "Q7: set model name to 'Ridge Regression'."
assert ridge_result["mae"] >= 0, "Q7: MAE should be non-negative."
assert ridge_result["rmse"] >= 0, "Q7: RMSE should be non-negative."
assert -1 <= ridge_result["r2"] <= 1, "Q7: R² should be in a reasonable range."
assert isinstance(ridge_alpha_results, pd.DataFrame), "Q7: ridge_alpha_results should be a DataFrame."
assert len(ridge_alpha_results) == 2, "Q7: compare exactly two alpha values."
assert len(ridge_comment.strip()) >= 15, "Q7: write a short interpretation sentence."
print("Q7 check passed.")



## Q8 [1 point]
Train and evaluate a **Random Forest regressor** inside a pipeline.

**Tasks**
- Create a pipeline called `rf_reg`
- Use:
  - `n_estimators=200`
  - `max_depth=12`
  - `random_state=RANDOM_STATE`
- Evaluate it and store the result in `rf_reg_result`


In [ ]:

# Q8 - YOUR CODE HERE

# TODO: create and evaluate the random forest regressor
rf_reg = None
rf_reg_metrics = None

rf_reg_result = {
    "model": "Random Forest Regressor",
    "mae": None,
    "rmse": None,
    "r2": None,
}


In [ ]:

# Q8 - quick check
assert isinstance(rf_reg_result, dict), "Q8: rf_reg_result should be a dictionary."
assert rf_reg_result["mae"] >= 0, "Q8: MAE should be non-negative."
assert rf_reg_result["rmse"] >= 0, "Q8: RMSE should be non-negative."
assert -1 <= rf_reg_result["r2"] <= 1, "Q8: R² should be in a reasonable range."
print("Q8 check passed.")


In [ ]:

regression_results_final = pd.concat(
    [
        regression_baselines,
        pd.DataFrame([ridge_result, rf_reg_result]),
    ],
    ignore_index=True,
).sort_values(by="r2", ascending=False)

regression_results_final



# Part C. Submission and reflection



## Q9 [1 point]
GitHub Classroom submission information.

Fill in:
- your GitHub Classroom repository URL
- your final commit hash

This question is worth **1 point** together with a successful upload to GitHub Classroom.


In [ ]:

# Q9 - YOUR ANSWER HERE

github_repo_url = "PASTE_YOUR_GITHUB_CLASSROOM_REPO_URL_HERE"
final_commit_hash = "PASTE_YOUR_FINAL_COMMIT_HASH_HERE"


In [ ]:

# Q9 - quick check
assert github_repo_url != "PASTE_YOUR_GITHUB_CLASSROOM_REPO_URL_HERE", "Q9: please replace the placeholder repo URL."
assert final_commit_hash != "PASTE_YOUR_FINAL_COMMIT_HASH_HERE", "Q9: please replace the placeholder commit hash."
print("Q9 check passed.")



## Q10 [1 point]
Assignment feedback.

In **3-5 sentences**, tell us:
- what felt easy,
- what felt difficult,
- and what could be improved in the notebook.


In [ ]:

# Q10 - YOUR ANSWER HERE

assignment_feedback = """
Write your feedback here.
"""


In [ ]:

# Q10 - quick check
assert len(assignment_feedback.strip()) >= 30, "Q10: please write a short feedback paragraph."
print("Q10 check passed.")



## Final reminder before submission

- Run **Kernel → Restart & Run All**
- Save the notebook
- Commit your changes
- Push to **GitHub Classroom**
